In [ ]:
#@title ## Extract and separate track music.
#@markdown Add information about the song.

#@markdown ---
#@markdown ### Enter a url:
url = "https://www.youtube.com/watch?v=w5iOeCaRagE" #@param {type:"string"}
#@markdown ### Enter a file name:
number_stems = "4" #@param [2, 4, 5]

In [45]:
import yt_dlp

def __check_status(d):
    if d['status'] == 'finished':
      filename = d['filename']
      print("{} downloading is done! Now to next step, converting ...".format(filename))

def __downloading(video_url, filename='filename', download=True):
    ydl_opts = {
          'format': 'bestaudio/best',
          'outtmpl': filename,
          'noplaylist': True,
          'quiet': True,
          'no_warnings': True,
          'postprocessors': [{
              'key': 'FFmpegExtractAudio',
              'preferredcodec': 'mp3',
              'preferredquality': '192',
          }],
          'progress_hooks': [__check_status]
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
         return ydl.extract_info(video_url, download=download)

def downloading_audio(video_url, filename="filename"):
    """
    Faz o download do áudio de um vídeo e retorna seu título.

    :param video_url: URL do vídeo.
    :param filename: Nome do arquivo (padrão: "filename").
    :return: Título do áudio baixado.
    """
    data = __downloading(video_url, filename)

    # Se 'entries' existir e for uma lista válida, pega o título do primeiro item
    if isinstance(data, dict):
        if "entries" in data and isinstance(data["entries"], list) and data["entries"]:
            title = data["entries"][0].get("title", "filename")
        else:
            title = data.get("title", "filename")
    else:
        title = "filename"  # Retorno seguro caso `data` não seja um dicionário

    return title

filename_music_src = downloading_audio(url)

[download] 100% of    4.96MiB in 00:00:00 at 22.44MiB/s  filename downloading is done! Now to next step, converting ...


In [46]:
print(filename_music_src)

import re

def format_string(texto):
    if not texto.strip():  # Se for vazio ou só tiver espaços, retorna vazio
        return ""
    texto = texto.lower()  # Converte para minúsculas
    texto = texto.replace(" ", "_")  # Substitui espaços por _
    texto = re.sub(r"[^\w]", "", texto)  # Remove caracteres não alfanuméricos
    return texto

filename_music = format_string(filename_music_src)
print(filename_music)

Até que o senhor venha | Attos2 Worship -  Thermut Lopes
até_que_o_senhor_venha__attos2_worship___thermut_lopes


In [47]:
import os

def rename_file(title_music: str, source_file: str = "filename.mp3"):
    """
    Renomeia um arquivo MP3 caso title_music seja diferente de "filename".

    :param title_music: Novo nome do arquivo (caso não seja "filename").
    :param source_file: Nome do arquivo original (padrão: "filename.mp3").
    """
    if title_music == "filename":
        print("Nenhuma renomeação necessária.")
        return

    destination_file = f"{title_music}.mp3"

    if not os.path.exists(source_file):
        print(f"Erro: O arquivo '{source_file}' não existe.")
        return

    try:
        os.rename(source_file, destination_file)
        print(f"Arquivo renomeado para: {destination_file}")
    except OSError as e:
        print(f"Erro ao renomear o arquivo: {e}")

# Exemplos de uso
rename_file(title_music=filename_music)  


Arquivo renomeado para: até_que_o_senhor_venha__attos2_worship___thermut_lopes.mp3


In [48]:
import spleeter


!spleeter separate -p spleeter:4stems -o output/ até_que_o_senhor_venha__attos2_worship___thermut_lopes.mp3

INFO:spleeter:File output/até_que_o_senhor_venha__attos2_worship___thermut_lopes/drums.wav written succesfully
INFO:spleeter:File output/até_que_o_senhor_venha__attos2_worship___thermut_lopes/bass.wav written succesfully
INFO:spleeter:File output/até_que_o_senhor_venha__attos2_worship___thermut_lopes/vocals.wav written succesfully
INFO:spleeter:File output/até_que_o_senhor_venha__attos2_worship___thermut_lopes/other.wav written succesfully


In [ ]:
# import spleeter

# print(filename)

# def separate(file_name, amount_steam=4, folder_output='output'):
#     if not file_name.strip():  # Se for vazio ou só tiver espaços, retorna vazio
#         raise ValueError("filename is empty.")
#     print("iniciando")
#     ! spleeter separate -p spleeter:4stems -o $folder_output/ $file_name

# separate(filename, 4, output_folder)

In [50]:
print(filename_music)

até_que_o_senhor_venha__attos2_worship___thermut_lopes


In [52]:
from pydub import AudioSegment

def convert_wav_to_mp3(directory: str, stems: int):
    """
    Converte arquivos .wav para .mp3 dentro de um diretório especificado.

    :param directory: Caminho do diretório onde estão os arquivos de áudio.
    :param stems: Número de stems a serem convertidos (2, 4 ou 5).
    :return: Lista de caminhos dos arquivos convertidos.
    """
    stem_files = ["vocals", "other"]
    
    if stems >= 4:
        stem_files.extend(["drums", "bass"])
    
    if stems == 5:
        stem_files.append("piano")

    mp3_files = []

    for stem in stem_files:
        wav_path = f"{directory}/{stem}.wav"
        mp3_path = f"{directory}/{stem}.mp3"

        sound = AudioSegment.from_wav(wav_path)
        sound.export(mp3_path, format="mp3")
        mp3_files.append(mp3_path)

    return mp3_files

output_dir_files = f"output/{filename_music}"
converted_files = convert_wav_to_mp3(directory=output_dir_files, stems=4)
print("Arquivos convertidos:", converted_files)


Arquivos convertidos: ['output/até_que_o_senhor_venha__attos2_worship___thermut_lopes/vocals.mp3', 'output/até_que_o_senhor_venha__attos2_worship___thermut_lopes/other.mp3', 'output/até_que_o_senhor_venha__attos2_worship___thermut_lopes/drums.mp3', 'output/até_que_o_senhor_venha__attos2_worship___thermut_lopes/bass.mp3']


In [53]:
import os
import zipfile

def create_zip(title_music: str):
    """
    Cria um arquivo ZIP contendo todos os arquivos .mp3 do diretório especificado.

    :param title_music: Nome do ZIP (se for "filename", mantém "filename.zip").
    :param output_dir: Diretório onde os arquivos .mp3 estão localizados.
    """
    # Define o nome do arquivo ZIP
    default_name = title_music if title_music != "filename" else "filename"
    output_dir = f"output/{default_name}"

    print(output_dir)
    # Verifica se o diretório existe
    if not os.path.exists(output_dir):
        print(f"Erro: O diretório '{output_dir}' não existe.")
        return

    # Lista apenas arquivos .mp3 no diretório
    mp3_files = [f for f in os.listdir(output_dir) if f.endswith(".mp3")]

    if not mp3_files:
        print("Nenhum arquivo .mp3 encontrado para compactar.")
        return

    zip_path = os.path.join(f"{default_name}.zip")

    print(zip_path)

    # Criando o ZIP
    try:
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
            for file in mp3_files:
                file_path = os.path.join(output_dir, file)
                zipf.write(file_path, arcname=file)  # Adiciona ao ZIP sem o caminho completo

        print(f"Arquivo ZIP criado: {zip_path}")
    except Exception as e:
        print(f"Erro ao criar o arquivo ZIP: {e}")

create_zip(title_music=filename_music)  


output/até_que_o_senhor_venha__attos2_worship___thermut_lopes
até_que_o_senhor_venha__attos2_worship___thermut_lopes.zip
Arquivo ZIP criado: até_que_o_senhor_venha__attos2_worship___thermut_lopes.zip
